# 1 Data Preparation

## Imports

In [1]:
import pandas as pd
from pathlib import Path
import os
pd.set_option('display.max_columns', None)

# Load & combine data

In [2]:
all_data = []

# Search for folders in /raw
data_dir = Path('../data/raw')
night_folders = sorted([d for d in os.listdir(data_dir) if (data_dir / d).is_dir()])

print(f"\nFound {len(night_folders)} nights:")
print(night_folders)

for night_id in night_folders:
    print(f"\nProcessing {night_id}...")
    
    # Paths
    accel_path = f'{data_dir}/{night_id}/Accelerometer.csv'
    gyro_path = f'{data_dir}/{night_id}/Gyroscope.csv'
    
    # Check if files exist
    if not os.path.exists(accel_path) or not os.path.exists(gyro_path):
        print(f"Missing files, skipping")
        continue
    
    # Load
    accel = pd.read_csv(accel_path)
    gyro = pd.read_csv(gyro_path)
    
    # Merge accelerometer and gyroscope on time column
    merged = accel.merge(
        gyro,
        on=['time', 'seconds_elapsed'],
        how='inner'
    )
    
    # add night_id
    merged['night_id'] = night_id
    
    # convert datetime 
    merged['time'] = pd.to_datetime(merged['time'])
    
    print(f"Loaded {len(merged)} rows")
    
    all_data.append(merged)

# Alle Nächte kombinieren
df = pd.concat(all_data, ignore_index=True)

print(f"\nTotal data: {len(df)} rows from {len(all_data)} nights")



Found 6 nights:
['2026-04-21_22-34-53', '2026-04-28_22-22-45', '2026-04-30_23-05-56', '2026-05-01_23-22-33', '2026-05-02_21-05-13', '2026-05-03_22-37-52']

Processing 2026-04-21_22-34-53...
Loaded 2564066 rows

Processing 2026-04-28_22-22-45...
Loaded 2700786 rows

Processing 2026-04-30_23-05-56...
Loaded 1629754 rows

Processing 2026-05-01_23-22-33...
Loaded 2944147 rows

Processing 2026-05-02_21-05-13...
Loaded 3332879 rows

Processing 2026-05-03_22-37-52...
Loaded 2467382 rows

Total data: 15639014 rows from 6 nights


## Load manual labels

In [3]:
data_dir = Path('../data')
labels = pd.read_csv(data_dir / 'raw/manual-labels.csv')

#Convert datetime
labels['bed_time'] = pd.to_datetime(labels['bed_time'], format='mixed')
labels['wake_time'] = pd.to_datetime(labels['wake_time'], format='mixed')

print("Labels loaded:")
print(labels)

Labels loaded:
              night_id            bed_time           wake_time
0  2026-04-21_22-34-53                 NaT                 NaT
1  2026-04-28_22-22-45 2026-04-29 00:24:00 2026-04-29 08:01:00
2  2026-04-30_23-05-56 2026-05-01 01:27:00 2026-05-01 08:12:00
3  2026-05-01_23-22-33 2026-05-02 01:49:00 2026-05-02 09:17:00
4  2026-05-02_21-05-13 2026-05-02 23:16:00 2026-05-03 08:00:00
5  2026-05-03_22-37-52 2026-05-04 01:01:00 2026-05-04 07:47:00


## Add manual labels to all_data

In [4]:
df['label'] = 'UNKNOWN'

for _, label_row in labels.iterrows():
    night_id = label_row['night_id']
    bed_time = label_row['bed_time']
    wake_time = label_row['wake_time']
    
    # Filter per night
    night_mask = df['night_id'] == night_id
    
    # SLEEP: time period between bed and awake
    sleep_mask = night_mask & (df['time'] >= bed_time) & (df['time'] <= wake_time)
    df.loc[sleep_mask, 'label'] = 'SLEEP'
    
    # AWAKE: time period before and after bed and awake
    awake_mask = night_mask & ((df['time'] < bed_time) | (df['time'] > wake_time))
    df.loc[awake_mask, 'label'] = 'AWAKE'
    
    print(f"  {night_id}: {sleep_mask.sum()} sleep, {awake_mask.sum()} awake")

  2026-04-21_22-34-53: 0 sleep, 0 awake
  2026-04-28_22-22-45: 1986769 sleep, 714017 awake
  2026-04-30_23-05-56: 870230 sleep, 759524 awake
  2026-05-01_23-22-33: 2080229 sleep, 863918 awake
  2026-05-02_21-05-13: 2567405 sleep, 765474 awake
  2026-05-03_22-37-52: 1761242 sleep, 706140 awake


## Check and save

In [ ]:
print(f"\nFinal label distribution:")
print(df['label'].value_counts())

print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Save???
df.to_parquet('../data/processed/all_data_with_labels.parquet', index=False)
print("\nSaved: processed/all_data_with_labels.parquet")


Final label distribution:
label
SLEEP      9265875
AWAKE      3809073
UNKNOWN    2564066
Name: count, dtype: int64

DataFrame shape: (15639014, 10)
Columns: ['time', 'seconds_elapsed', 'z_x', 'y_x', 'x_x', 'z_y', 'y_y', 'x_y', 'night_id', 'label']

Saved: processed/all_data_with_labels.parquet
